# CSC230 SQLite Activity Notebook

This notebook is the main in-class path. It uses `sqlite3` from the Python standard library, so there is no MySQL server setup step.

In [ ]:
# Config cell
DB_PATH = "classroom.db"

In [ ]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

def run_sql(sql: str):
    try:
        cur = conn.execute(sql)
        if sql.lstrip().upper().startswith(("SELECT", "PRAGMA")):
            rows = cur.fetchall()
            print(rows)
        else:
            conn.commit()
            print("OK")
    except sqlite3.Error as err:
        print(f"SQLite error: {err}")


## Sanity checks

In [ ]:
run_sql("SELECT sqlite_version();")
run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

## Reset cell (safe rerun)
Drops in dependency order so you can rerun activity cells cleanly.

In [ ]:
run_sql("DROP TABLE IF EXISTS enrollments;")
run_sql("DROP TABLE IF EXISTS students;")
run_sql("DROP TABLE IF EXISTS courses;")

## Cell A: Create 2 relations + constraints (PK, NOT NULL, DEFAULT)

In [ ]:
run_sql("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    class_year INTEGER NOT NULL DEFAULT 2026
);
""")

run_sql("""
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL DEFAULT 1
);
""")

## Cell B: Add a foreign key using REFERENCES

In [ ]:
run_sql("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL REFERENCES students(student_id),
    course_id INTEGER NOT NULL REFERENCES courses(course_id),
    status TEXT NOT NULL DEFAULT 'enrolled'
);
""")

## Cell C: Insert one valid row and one FK-violating row

In [ ]:
run_sql("INSERT INTO students(student_id, full_name) VALUES (1, 'Ada Lovelace');")
run_sql("INSERT INTO courses(course_id, title, credits) VALUES (10, 'Databases', 4);")
run_sql("INSERT INTO enrollments(enrollment_id, student_id, course_id) VALUES (100, 1, 10);")
run_sql("INSERT INTO enrollments(enrollment_id, student_id, course_id) VALUES (101, 999, 10);")  # should fail FK
run_sql("SELECT * FROM enrollments ORDER BY enrollment_id;")

## Cell D: ALTER TABLE ... ADD ... DEFAULT

In [ ]:
run_sql("ALTER TABLE students ADD COLUMN email TEXT NOT NULL DEFAULT 'unknown@school.edu';")
run_sql("PRAGMA table_info(students);")

## Common errors

- `no such table`: Run the reset cell, then run cells A -> D in order.
- `FOREIGN KEY constraint failed`: Expected for the deliberate bad insert in Cell C.
- `table ... already exists`: Run the reset cell and retry.